In [ ]:
import pandas as pd

from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.graph_objects as go
pio.renderers.default = "notebook"
# pio.renderers.default = "browser"

In [ ]:
# CSVデータの読み込み
csv_path = r"/home/raspi5_16gb/projects/driving_robot/tests/research/results/drive_log_real_20260916_141206.csv"
df = pd.read_csv(csv_path)

print(f"行数: {len(df)}")
print(f"NaNを含む行数: {df.isna().any(axis=1).sum()}")
# ref_speed_kmh / deviation_kmh / accel_ff_pct / brake_ff_pct は
# PRE_DRIVE_CHECK 区間（パターン未ロード）でNaNになるのが正常なので、行削除はしない

df.head()

In [ ]:
# インタラクティブなプロット（plotly）
# 1軸目: 基準車速・実車速 / 2軸目: 偏差 / 3軸目: アクセル・ブレーキ開度（FF・実開度）
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    row_heights=[0.4, 0.25, 0.35],
    subplot_titles=("車速（基準 / 実測）", "偏差", "アクセル・ブレーキ開度"),
)

# 1軸目: 基準車速・実車速
fig.add_trace(
    go.Scatter(x=df["elapsed_s"], y=df["ref_speed_kmh"], name="基準車速", line=dict(color="royalblue")),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=df["elapsed_s"], y=df["actual_speed_kmh"], name="実車速", line=dict(color="firebrick")),
    row=1, col=1,
)

# 2軸目: 偏差
fig.add_trace(
    go.Scatter(x=df["elapsed_s"], y=df["deviation_kmh"], name="偏差", line=dict(color="seagreen")),
    row=2, col=1,
)
fig.add_hline(y=0, line=dict(color="gray", dash="dot"), row=2, col=1)

# 3軸目: アクセル・ブレーキ開度（FF=破線、実開度=実線）
fig.add_trace(
    go.Scatter(x=df["elapsed_s"], y=df["accel_ff_pct"], name="アクセル開度FF[%]", line=dict(color="orange", dash="dash")),
    row=3, col=1,
)
fig.add_trace(
    go.Scatter(x=df["elapsed_s"], y=df["brake_ff_pct"], name="ブレーキ開度FF[%]", line=dict(color="dodgerblue", dash="dash")),
    row=3, col=1,
)
fig.add_trace(
    go.Scatter(x=df["elapsed_s"], y=df["accel_actual_pct"], name="アクセル実開度[%]", line=dict(color="orange")),
    row=3, col=1,
)
fig.add_trace(
    go.Scatter(x=df["elapsed_s"], y=df["brake_actual_pct"], name="ブレーキ実開度[%]", line=dict(color="dodgerblue")),
    row=3, col=1,
)

fig.update_yaxes(title_text="車速 [km/h]", row=1, col=1)
fig.update_yaxes(title_text="偏差 [km/h]", row=2, col=1)
fig.update_yaxes(title_text="開度 [%]", row=3, col=1)
fig.update_xaxes(title_text="経過時間 [s]", row=3, col=1)
fig.update_xaxes(rangeslider=dict(visible=True), row=3, col=1)

fig.update_layout(
    height=900,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    title="走行ログ インタラクティブ分析",
)

fig.show()
# fig.write_html("result.html")